get and load the data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!wget --no-check-certificate -O "/content/drive/MyDrive/Colab Notebooks/Machine Learning/Chess Engine/Datasets/month_pgn_files.zip" "https://database.nikonoel.fr/lichess_elite_2021-12.zip"

--2024-12-24 12:28:16--  https://database.nikonoel.fr/lichess_elite_2021-12.zip
Resolving database.nikonoel.fr (database.nikonoel.fr)... 178.162.201.225
Connecting to database.nikonoel.fr (database.nikonoel.fr)|178.162.201.225|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 133544685 (127M) [application/zip]
Saving to: ‘/content/drive/MyDrive/Colab Notebooks/Machine Learning/Chess Engine/Datasets/month_pgn_files.zip’

/content/drive/MyDr 100%[===================>] 127.36M  13.2MB/s    in 11s     

2024-12-24 12:28:30 (11.1 MB/s) - ‘/content/drive/MyDrive/Colab Notebooks/Machine Learning/Chess Engine/Datasets/month_pgn_files.zip’ saved [133544685/133544685]



In [ ]:
import zipfile

zip_file_path = '/content/drive/MyDrive/Colab Notebooks/Machine Learning/Chess Engine/Datasets/month_pgn_files.zip'

extract_folder = '/content/drive/MyDrive/Colab Notebooks/Machine Learning/Chess Engine/Datasets/lichess_elite_database/'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

print("ZIP file extracted successfully!")

ZIP file extracted successfully!


In [ ]:
!pip install chess

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.5/156.5 kB 1.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.1-py3-none-any.whl size=148497 sha256=6e2680ed5e0ab3deba0e05d844c0e825e61d9b81c0eef339144708a884678a84
  Stored in directory: /root/.cache/pip/wheels/2e/2d/23/1bfc95db984ed3ecbf6764167dc7526d0ab521cf9a9852544e
Successfully built chess


In [ ]:
# Function to load PGN file and return a list of games
def load_pgn_file(file_path, limit=None):
    games = []
    with open(file_path, 'r') as pgn_file:
        while True:
            game = pgn.read_game(pgn_file)
            if game is None:
                break
            games.append(game)
            if limit and len(games) >= limit:
                break
    return games


In [ ]:
# Directory where the PGN files are located
pgn_directory = '/content/drive/MyDrive/Colab Notebooks/Machine Learning/Chess Engine/Datasets/lichess_elite_database/'

# Get the list of files in the directory
files = os.listdir(pgn_directory)

# Limit number of files to process (e.g., 24 files)
LIMIT_OF_FILES = min(len(files), 24)

games = []

# Process each file in the directory
i = 1
for file in tqdm(files):
    if i > LIMIT_OF_FILES:
        break

    file_path = os.path.join(pgn_directory, file)

    if file.endswith('.pgn'):  # Ensure the file is a PGN file
        games.extend(load_pgn_file(file_path,limit=1000))

    i += 1

print(f"Loaded {len(games)} games.")


100%|██████████| 1/1 [00:05<00:00,  5.03s/it]

Loaded 1000 games.


preprocessing the data to get input and output

In [ ]:
import numpy as np
from chess import Board

In [ ]:
X, y = create_input_for_nn(games)
print(X.shape)
print(y.shape)

(84073, 13, 8, 8)
(84073,)


In [ ]:
y, move_to_int = encode_moves(y)
num_classes = len(move_to_int)
print(num_classes)

1802


building and training the model

In [ ]:
import tensorflow
print("GPU available:", tensorflow.config.list_physical_devices('GPU'))

GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
from tensorflow import keras
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Conv2D, Flatten, Dense, Input

In [ ]:
y = to_categorical(y, num_classes=len(move_to_int))
X = np.array(X)

In [ ]:
model = Sequential([
    Input(shape=(13, 8, 8)),
    Conv2D(64, (3, 3), activation='relu'),
    Conv2D(128, (3, 3), activation='relu'),
    Flatten(),
    Dense(256, activation='relu'),
    Dense(num_classes, activation='softmax')
])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 11, 6, 64)           │           4,672 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 9, 4, 128)           │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 4608)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │       1,179,904 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1802)                │         463,114 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,721,546 (6.57 MB)

 Trainable params: 1,721,546 (6.57 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history=model.fit(X, y, epochs=50, validation_split=0.1, batch_size=64)

Epoch 1/50
1183/1183 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - accuracy: 0.0350 - loss: 6.2713 - val_accuracy: 0.0874 - val_loss: 5.5411
Epoch 2/50
1183/1183 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.0981 - loss: 5.2620 - val_accuracy: 0.1086 - val_loss: 5.1719
Epoch 3/50
1183/1183 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.1406 - loss: 4.6029 - val_accuracy: 0.1284 - val_loss: 5.0549
Epoch 4/50
1183/1183 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.1851 - loss: 4.0202 - val_accuracy: 0.1282 - val_loss: 5.0345
Epoch 5/50
1183/1183 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.2328 - loss: 3.5659 - val_accuracy: 0.1352 - val_loss: 5.1836
Epoch 6/50
1183/1183 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.2850 - loss: 3.1487 - val_accuracy: 0.1321 - val_loss: 5.3401
Epoch 7/50
1183/1183 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.3419 - loss: 2.7837 - val_accuracy: 0.1351 - val_loss: 5.7046
Epoch 8/50
1183/1183 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.3939 - loss: 2.4841 

KeyboardInterrupt: 

# Streaming and Batch Traning

import the modules

In [2]:
!pip install chess

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.5/156.5 kB 9.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.1-py3-none-any.whl size=148497 sha256=f0c7e7e92f05b274f39cb29c0ac98a1090cd79e1fdc9817089233a54f5c6565c
  Stored in directory: /root/.cache/pip/wheels/2e/2d/23/1bfc95db984ed3ecbf6764167dc7526d0ab521cf9a9852544e
Successfully built chess


In [ ]:
import tensorflow
print("GPU available:", tensorflow.config.list_physical_devices('GPU'))

GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
import tensorflow

In [4]:
from tensorflow import keras
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Conv2D, Flatten, Dense, Input, MaxPooling2D, Dropout
from keras.utils import to_categorical
from keras.models import load_model
import os
import chess
from chess import pgn
from tqdm import tqdm
import pickle
import numpy as np
import pandas as pd

define all the categories

In [ ]:
files = [file for file in os.listdir('/content/drive/MyDrive/Colab Notebooks/Machine Learning/Chess Engine/Datasets/lichess_elite_database') if file.endswith('.pgn')]
len(files)

1

In [ ]:
map_location = "/content/drive/MyDrive/Colab Notebooks/Machine Learning/Chess Engine/Notebooks/move_to_int.pkl"
def load_mapping():
  try:
    with open(map_location, 'rb') as f:
            mapping = pickle.load(f)
  except FileNotFoundError:
        mapping = {}
  return mapping

def save_mapping(mapping):
    with open(map_location, 'wb') as f:
        pickle.dump(mapping, f)

def update_mapping(game, move_to_int):
    board = game.board()
    for move in game.mainline_moves():
        move_uci = move.uci()
        if move_uci not in move_to_int:
            # If the move is not already in the mapping, add it
            move_to_int[move_uci] = len(move_to_int)
    return move_to_int

def load_game_from_file(file_path):
    with open(file_path, 'r') as file:
        game = pgn.read_game(file)
    return game

In [ ]:
move_to_int = load_mapping()
game_count = 0
for file_name in files:
    file_path = os.path.join('/content/drive/MyDrive/Colab Notebooks/Machine Learning/Chess Engine/Datasets/lichess_elite_database', file_name)
    with open(file_path, 'r') as file:
        while True:
            game = pgn.read_game(file)
            if game is None:
                break
            move_to_int = update_mapping(game, move_to_int)
            game_count += 1
            if game_count % 1000 == 0:
                save_mapping(move_to_int)
                print(f"Total games processed: {game_count}")
                print(f"Total unique moves: {len(move_to_int)}")
                print("Mapping saved.")

save_mapping(move_to_int)
print(f"Total games processed: {game_count}")
print(f"Total unique moves: {len(move_to_int)}")
print("Mapping saved.")

preprocess and train the model using mini-batch approach

In [ ]:
def stream_pgn_file(file_path):
    with open(file_path, 'r') as pgn_file:
        while True:
            game = pgn.read_game(pgn_file)
            if game is None:
                break
            yield game

In [ ]:
def board_to_matrix(board: chess.Board):
    # 8x8 is a size of the chess board.
    # 12 = number of unique pieces.
    # 13th board for legal moves (WHERE we can move)
    matrix = np.zeros((13, 8, 8))
    piece_map = board.piece_map()

    # Populate first 12 8x8 boards (where pieces are)
    for square, piece in piece_map.items():
        row, col = divmod(square, 8)
        piece_type = piece.piece_type - 1
        piece_color = 0 if piece.color else 6
        matrix[piece_type + piece_color, row, col] = 1

    # Populate the legal moves board (13th 8x8 board)
    legal_moves = board.legal_moves
    for move in legal_moves:
        to_square = move.to_square
        row_to, col_to = divmod(to_square, 8)
        matrix[12, row_to, col_to] = 1

    return matrix

def create_input_for_nn(games):
    X = []
    y = []
    for game in games:
        board = game.board()
        for move in game.mainline_moves():
            X.append(board_to_matrix(board))
            y.append(move.uci())
            board.push(move)
    return np.array(X, dtype=np.int32), np.array(y)

In [ ]:
def process_and_train(file_path, model, batch_size=1000, checkpoint_dir="checkpoints", save_interval=10):
    """
    Train a model on batches of games streamed from a PGN file, saving checkpoints every `save_interval` epochs.
    Args:
        file_path (str): Path to the PGN file.
        model (tf.keras.Model): The neural network model to train.
        batch_size (int): Number of games to process in each batch.
        checkpoint_dir (str): Directory to save model checkpoints.
        save_interval (int): Save the model after every `save_interval` epochs.
    """
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)

    game_batch = []
    game_count = 0
    epoch_count = 0  # Keep track of the total epochs
    move_to_int = load_mapping()

    for game in stream_pgn_file(file_path):
        game_batch.append(game)
        game_count += 1

        if game_count % batch_size == 0:
            # Prepare data
            X, y = create_input_for_nn(game_batch)
            y = [move_to_int[move] for move in y]
            y = to_categorical(y, num_classes=len(move_to_int))

            # Train the model
            model.fit(X, y, epochs=1)  # Train on the batch
            epoch_count += 1

            # Save checkpoint if required
            if epoch_count % save_interval == 0:
                checkpoint_path = os.path.join(checkpoint_dir, f"model_epoch_{epoch_count}.keras")
                model.save(checkpoint_path)
                print(f"Model saved at epoch {epoch_count} to {checkpoint_path}")

            print(f"Processed {game_count} games...")
            # Clear memory
            del game_batch[:]
            game_batch = []

    # Process remaining games if any
    if game_batch:
        print("Processing final batch...")
        X, y = create_input_for_nn(game_batch)
        y = [move_to_int[move] for move in y]
        y = to_categorical(y, num_classes=len(move_to_int))
        model.fit(X, y, epochs=1)
        epoch_count += 1

        # Save final checkpoint
        checkpoint_path = os.path.join(checkpoint_dir, f"model_epoch_{epoch_count}.h5")
        model.save(checkpoint_path)
        print(f"Final model saved at epoch {epoch_count} to {checkpoint_path}")

    print("Training completed.")

In [ ]:
#load the map
move_to_int=load_mapping()

#define the model
cnn_model = Sequential()

# Convolutional Layers
cnn_model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(13, 8, 8)))
cnn_model.add(Conv2D(64, (3, 3), activation='relu'))
cnn_model.add(Conv2D(128, (3, 3), activation='relu'))

# Flatten the output from Conv2D layers to feed into Dense layers
cnn_model.add(Flatten())

# Fully Connected (Dense) Layers
cnn_model.add(Dense(512, activation='relu'))
cnn_model.add(Dropout(0.5))  # Dropout for regularization

cnn_model.add(Dense(256, activation='relu'))
cnn_model.add(Dropout(0.5))

# Output Layer
cnn_model.add(Dense(len(move_to_int), activation='softmax'))  # Output size = number of possible moves

# Compile the model
cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'], validation_split=0.1)
cnn_model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_18 (Conv2D)                   │ (None, 11, 6, 32)           │           2,336 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_19 (Conv2D)                   │ (None, 9, 4, 64)            │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_20 (Conv2D)                   │ (None, 7, 2, 128)           │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_6 (Flatten)                  │ (None, 1792)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_17 (Dense)                     │ (None, 512)                 │         918,016 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_12 (Dropout)                 │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_18 (Dense)                     │ (None, 256)                 │         131,328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_13 (Dropout)                 │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_19 (Dense)                     │ (None, 1941)                │         498,837 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,642,869 (6.27 MB)

 Trainable params: 1,642,869 (6.27 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
process_and_train(
    file_path='/content/drive/MyDrive/Colab Notebooks/Machine Learning/Chess Engine/Datasets/lichess_elite_database/lichess_elite_2021-12.pgn',
    model=cnn_model,
    batch_size=1000,
    checkpoint_dir='/content/drive/MyDrive/Colab Notebooks/Machine Learning/Chess Engine/Notebooks',
    save_interval=100
)

2628/2628 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - accuracy: 0.0191 - loss: 6.5273
Processed 1000 games...
2592/2592 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0.0447 - loss: 5.9361
Processed 2000 games...
2536/2536 ━━━━━━━━━━━━━━━━━━━━ 15s 6ms/step - accuracy: 0.0584 - loss: 5.7734
Processed 3000 games...
2573/2573 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.0651 - loss: 5.6981
Processed 4000 games...
2529/2529 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.0689 - loss: 5.6256
Processed 5000 games...
2603/2603 ━━━━━━━━━━━━━━━━━━━━ 24s 9ms/step - accuracy: 0.0743 - loss: 5.6074
Processed 6000 games...
2645/2645 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.0776 - loss: 5.5848
Processed 7000 games...
2537/2537 ━━━━━━━━━━━━━━━━━━━━ 18s 7ms/step - accuracy: 0.0782 - loss: 5.5502
Processed 8000 games...
2605/2605 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.0799 - loss: 5.5329
Processed 9000 games...
2584/2584 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.0807 - loss: 5.5052
Process

Model saved at epoch 100 to /content/drive/MyDrive/Colab Notebooks/Machine Learning/Chess Engine/Notebooks/model_epoch_100.h5
Processed 100000 games...
2561/2561 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.0863 - loss: 5.4413
Processed 101000 games...
2624/2624 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.0803 - loss: 5.4711
Processed 102000 games...
2553/2553 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.0823 - loss: 5.4309
Processed 103000 games...


KeyboardInterrupt: 

In [5]:
# List of imported libraries and their respective version attributes
libraries = {
    "numpy": "np",
    "tensorflow": "tensorflow",
    "keras": "keras",
    "chess": "chess",
    "tqdm": "tqdm",
    "pickle": "pickle",  # pickle doesn't have a version attribute
    "pandas": "pd"
}

# Function to fetch and print versions
def get_versions():
    for lib_name, lib_alias in libraries.items():
        try:
            if lib_name == "pickle":
                print(f"{lib_name}: Standard Library (No version attribute)")
            else:
                lib = eval(lib_alias)
                print(f"{lib_name}: {lib.__version__}")
        except AttributeError:
            print(f"{lib_name}: Version not available")
        except NameError:
            print(f"{lib_name}: Not imported")

# Call the function
get_versions()


numpy: 1.26.4
tensorflow: 2.17.1
keras: 3.5.0
chess: 1.11.1
tqdm: Version not available
pickle: Standard Library (No version attribute)
pandas: 2.2.2


In [ ]:
!python --version

Python 3.10.12
